<!-- OPS:MAT-02:inicio -->
## Esto es “completo” hoy (55 min)

Marca con lo que realmente ocurrió. No hace falta terminar todo.

- [ ] El entorno funciona o abrí `la ruta de rescate docente (no incluida)` y lo declaré.
- [ ] Producto mínimo: Un placebo o control negativo y una frase que diga qué no demuestra sobre el supuesto no observable.
- [ ] Si algo falló: una frase del error, no el traceback completo.
- [ ] Bitácora escrita, aunque diga `lab incompleto`.

Completo ≠ entendido del todo. Incompleto ≠ no puedes seguir. Si la instalación falla después de 10 minutos, usa `la ruta de rescate docente (no incluida)`, conserva el artefacto y marca incompleto.
**Extensión:** Si queda tiempo: Qini, varios placebos, E-value y R² parcial.
<!-- OPS:MAT-02:fin -->

# L11 · Validación: Qini, placebo y sensibilidad

**S11 · Lectura: Cap. 14 · 40–55 min**

## La pregunta

¿Qué prueba podría hacer que dejemos de confiar en el estimador?

## Qué harás

Validar un efecto heterogéneo sin confundir un placebo con una prueba total.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Qini, placebo, control negativo y sensibilidad del estimador A2.

> **Idea clave:** Validar no es buscar una cifra bonita; es buscar fallos plausibles.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
from lab_assumptions import rr_from_d
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor
rng = setup()
N_PERM = R(30, 200)
print("permutaciones del placebo:", N_PERM)

## 1. Evaluar un CATE: el problema (§14.1–14.2)
Con datos de un **experimento** podemos evaluar un ranking de CATE sin conocer $\tau_i$: el *resultado transformado*
$$Y^*_i=Y_i\,\frac{T_i-e}{e(1-e)}\quad\Rightarrow\quad E[Y^*\mid X]=\tau(X)$$
(con $e$ conocido). Ordenamos las unidades de mayor a menor $\hat\tau$, acumulamos $Y^*$ y comparamos con la línea del **azar**: la curva de **Qini**.

Mundo simulado (el del capítulo): $\tau(X)=2+3X_0-X_1^2$, experimento con $e=0{,}5$.

In [ ]:
def make_dgp(n=3000, p=10, rng=None):
    rng = rng or np.random.default_rng(0)
    X = rng.normal(size=(n, p)); tau = 2 + 3 * X[:, 0] - X[:, 1] ** 2
    t = rng.binomial(1, .5, n); mu0 = 1.0 * X[:, 0] + X[:, 2] + .5 * X[:, 3] ** 2
    y = mu0 + t * tau + rng.normal(size=n)
    return X, y, t, tau, mu0
X, Y, T, tau, mu0 = make_dgp(3000, 10, np.random.default_rng(1))
tr, te = np.arange(2000), np.arange(2000, 3000)
mk = lambda: ExtraTreesRegressor(n_estimators=60, min_samples_leaf=10, max_features=0.8, random_state=0)
# T-learner entrenado SOLO en train; se evalúa en test
tau_hat = mk().fit(X[tr][T[tr] == 1], Y[tr][T[tr] == 1]).predict(X[te]) - mk().fit(X[tr][T[tr] == 0], Y[tr][T[tr] == 0]).predict(X[te])
print("τ̂ en test: media", tau_hat.mean().round(2), "· desv.", tau_hat.std().round(2))

### Completa tú: el área de Qini
`qini_area(tau_hat, y, t, e)` debe: (1) calcular $Y^*$; (2) ordenar por $\hat\tau$ **descendente**; (3) acumular $Y^*$ y dividir por $n$ (uplift acumulado); (4) restar la **línea del azar** (de 0 al total, proporcional a la fracción tratada); (5) devolver el promedio de esa diferencia (área entre la curva y el azar).

In [ ]:
# ✏️ Completa tú — qini_area
# Pista: Un buen ranking pone arriba a quienes más se benefician: la curva queda POR ENCIMA del azar.
def qini_area(tau_hat, y, t, e=0.5):
    """Área (promedio) entre la curva de Qini y la línea del azar."""
    y, t = np.asarray(y, float), np.asarray(t, float)
    # TODO 1: resultado transformado y* = y (t-e) / (e (1-e))
    # TODO 2: ordena por tau_hat descendente y acumula y* / n
    # TODO 3: línea del azar = fracción acumulada * total
    raise NotImplementedError("completa qini_area")

In [ ]:
r_ = np.random.default_rng(0); t_s = r_.binomial(1, .5, 2000); y_s = r_.normal(size=2000) + t_s * 2 * (np.arange(2000) < 1000)
bueno = (np.arange(2000) < 1000).astype(float)            # ranking perfecto: los 1000 primeros tienen efecto 2
lab_check("qini_area (propiedades)", lambda: (qini_area(bueno, y_s, t_s) > 0.2, qini_area(-bueno, y_s, t_s) < -0.2,
          abs(qini_area(r_.normal(size=2000), y_s, t_s)) < 0.25), [((), (True, True, True), 0)])

In [ ]:
q_modelo = qini_area(tau_hat, Y[te], T[te])
q_azar = np.mean([qini_area(np.random.default_rng(s).normal(size=len(te)), Y[te], T[te]) for s in range(50)])
print(f"Qini del modelo: {q_modelo:.3f} · Qini de un ranking al azar (50 repeticiones): {q_azar:.3f}")
def curva(score):
    ys = Y[te] * (T[te] - .5) / .25; o = np.argsort(-score); return np.cumsum(ys[o]) / len(te)
f = np.arange(1, len(te) + 1) / len(te)
plt.plot(f, curva(tau_hat), label="modelo"); plt.plot(f, f * curva(tau_hat)[-1], "k--", label="azar")
plt.xlabel("fracción tratada (de mayor a menor τ̂)"); plt.ylabel("uplift acumulado"); plt.legend(); plt.show()

## 2. Baselines obligatorios (§14.3)
Un CATE solo "vale" si supera alternativas triviales:
1. **Efecto constante**: todos reciben el ATE ⇒ ranking sin información (área ≈ 0).
2. **Solo resultado**: ordenar por $\hat\mu_0(X)$ (quién tendría el mejor resultado sin tratamiento); es lo que hace un modelo predictivo sin causalidad.

## Sección de evaluación reservada

La edición pública termina antes de las celdas docentes que comparan con verdad simulada, efectos potenciales o resultados oráculo. Esa revisión se ejecuta aparte; aquí debes sostener la decisión con observables, incertidumbre y supuestos declarados.
